# S6E9 · 06：回到 04，只改变目标编码平滑强度

**操作：把本文件放进原 Kaggle 项目，用成功运行 04/05 的 Python 内核，依次运行第 1–4 节即可。**

**“Run All / 运行全部”会开始一套新的 LightGBM 五折训练。无需打开训练开关。** 不训练 CatBoost，不重跑旧 LightGBM，不需要重新执行 03、04、05。不会自动安装软件或提交 Kaggle。

本轮唯一新的模型设定：**目标编码 `smoothing: 1.0 → 10.0`**。恢复并固定 04 的深度 **5**（不是 05 的 6），固定 **50% 旧 CatBoost + 50% 新 LightGBM**，不调整融合权重。

| 已完成实验（来自你提供的截图） | 开发 OOF（约） | 已报告公榜 |
|---|---:|---:|
| 04：旧 CatBoost / 旧 LightGBM，50/50 | 0.94598624 | **0.94618** |
| 05：同样两种旧模型，25/75 | 0.946021 | 0.94615 |

这些是历史结果，**不是 06 的成绩**。本轮不重复选择 25/75，也不删除或覆盖 04/05 的文件。旧方案作为开发对照保留；公榜反馈已经影响过实验选择，因此后续 OOF 不能当成独立审计。

代码已完成语法、结构、小型合成数据流程测试；**没有使用你的真实竞赛 CSV 完成 06 训练，不保证提分。**


## 本轮为什么改这个？（无需操作）

现有实现中的目标编码是：

`编码值 = (该取值的正样本数 + smoothing × 训练子集正样本比例) / (该取值的样本数 + smoothing)`

把 `smoothing` 提高，会让样本较少的取值更靠近训练子集的整体均值，降低对少量标签统计的依赖。它也可能压低有用信号，所以 `10` 是一个实验候选，不是已证实的最佳值。没有证据仅凭 05 的公榜小幅下降就诊断原模型过拟合。

保留原来的内层五折交叉拟合；外层验证/测试的目标编码映射只用外层训练数据。原始数值、数字位特征、频数特征、类别映射和随机种子都不变。平滑 `10` 会用于原来所有目标编码列，不是只用于收入。

| 参数 | 04 | 06 |
|---|---:|---:|
| 目标编码 smoothing | 1.0 | **10.0** |
| max_depth / num_leaves | 5 / 31 | 5 / 31 |
| n_estimators / learning_rate | 5000 / 0.05 | 5000 / 0.05 |
| min_child_samples / reg_lambda | 100 / 5.0 | 100 / 5.0 |
| 早停等待 | 200 | 200 |
| 融合 CatBoost / LightGBM | 50% / 50% | 50% / 50% |

参考：[目标编码平滑](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.TargetEncoder.html)、[内部交叉拟合](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_target_encoder_cross_val.html)、[LightGBM 早停](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.early_stopping.html)。代码沿用本会话 04/05 的自定义编码，不改为 sklearn 的自动平滑编码器。


## 1. 加载函数（不训练）

独立使用：此单元格包含所需函数，不执行或导入旧 notebook。

In [1]:
from __future__ import annotations
import gc
import hashlib
import importlib.metadata
import json
import sys
import types
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from IPython.display import display

TARGET = "Will_Buy_EV"
NUMERIC = ["Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",
           "Charging_Stations_Near_Home", "Charging_Stations_Near_Work",
           "Environmental_Concern_Level"]


def features(frame, model_name):
    """Row-local features only: no labels or full-data target statistics."""
    x = frame.drop(columns=["id", TARGET], errors="ignore").copy()
    for col, scale, positions in [("Age", 1, 2), ("Annual_Income_USD", 1, 6),
                                 ("Daily_Commute_km", 10, 3),
                                 ("Charging_Stations_Near_Home", 1, 2),
                                 ("Charging_Stations_Near_Work", 1, 2)]:
        values = np.rint(frame[col].to_numpy() * scale).astype(np.int64)
        for digit in range(positions):
            x[f"{col}_digit_{digit}"] = (values // 10**digit % 10).astype(np.int8)
    income = np.rint(frame["Annual_Income_USD"]).astype(np.int64)
    x["income_last_2"] = income % 100
    x["income_last_3"] = income % 1000
    if model_name == "catboost":
        for col in NUMERIC:
            scale = 10 if col == "Daily_Commute_km" else 1
            x[f"{col}_as_category"] = np.rint(frame[col] * scale).astype(np.int64).astype(str)
    else:
        x["income_prefix_100"] = income // 100
        x["income_prefix_1000"] = income // 1000
    return x

def encode_fold(train_x, train_y, valid_x, test_x, seed, smoothing=1.0):
    """Inner OOF for training TE; outer-training-only maps for validation/test."""
    a, b, c = [d.copy().reset_index(drop=True) for d in (train_x, valid_x, test_x)]
    labels = np.asarray(train_y)
    inner = list(StratifiedKFold(5, shuffle=True, random_state=seed).split(a, labels))
    cats = a.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    keys = NUMERIC + cats + ["income_prefix_100", "income_prefix_1000"]
    for col in keys:
        encoded = np.full(len(a), np.nan)
        for fit_idx, hold_idx in inner:
            subset = pd.DataFrame({"key": a[col].iloc[fit_idx].to_numpy(), "y": labels[fit_idx]})
            stats = subset.groupby("key")["y"].agg(["sum", "count"])
            prior = labels[fit_idx].mean()
            mapping = (stats["sum"] + smoothing * prior) / (stats["count"] + smoothing)
            encoded[hold_idx] = a[col].iloc[hold_idx].map(mapping).fillna(prior).to_numpy()
        stats = pd.DataFrame({"key": a[col], "y": labels}).groupby("key")["y"].agg(["sum", "count"])
        prior = labels.mean()
        mapping = (stats["sum"] + smoothing * prior) / (stats["count"] + smoothing)
        a[f"{col}_te"] = encoded
        b[f"{col}_te"] = b[col].map(mapping).fillna(prior).to_numpy()
        c[f"{col}_te"] = c[col].map(mapping).fillna(prior).to_numpy()
        assert np.isfinite(encoded).all()
    # Label-free frequency: exclude a training row's own count contribution.
    counts = a["Annual_Income_USD"].value_counts()
    a["income_frequency"] = (a["Annual_Income_USD"].map(counts) - 1) / (len(a) - 1)
    for frame in (b, c):
        frame["income_frequency"] = frame["Annual_Income_USD"].map(counts).fillna(0) / len(a)
    for col in cats:
        dtype = pd.CategoricalDtype(categories=pd.unique(a[col]))
        for frame in (a, b, c):
            frame[col] = frame[col].astype(dtype)
    return a, b, c

def require(condition, message):
    if not bool(condition):
        raise ValueError(message)

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def input_hashes(project):
    root = Path(project) / "data"
    names = ["train.csv", "test.csv", "sample_submission.csv"]
    return {name: file_hash(root / name) for name in names if (root / name).is_file()}

def probabilities(values, expected_length, description):
    p = np.asarray(values, dtype=float)
    require(p.shape == (expected_length,), f"{description}: wrong prediction shape")
    require(np.isfinite(p).all(), f"{description}: NaN or infinity")
    require(((p >= 0) & (p <= 1)).all(), f"{description}: probabilities outside [0, 1]")
    return p

def align_csv(path, ids, columns):
    """Do not blend CSVs by their incidental row order: align explicitly by id."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"缺少文件：{path}。请先确认原 outputs 目录；尚未开始新训练。")
    frame = pd.read_csv(path)
    require(set(["id", *columns]).issubset(frame.columns), f"{path.name}: missing columns")
    require(frame.id.notna().all() and frame.id.is_unique, f"{path.name}: invalid/duplicate ids")
    wanted = pd.Index(np.asarray(ids))
    require(len(frame) == len(wanted) and wanted.isin(frame.id).all(), f"{path.name}: id sets differ")
    return frame.set_index("id").loc[wanted, columns].reset_index()

def load_project(project):
    project = Path(project)
    for name in ("train.csv", "test.csv"):
        if not (project / "data" / name).is_file():
            raise FileNotFoundError(f"找不到 {project / 'data' / name}；请检查 PROJECT。")
    train = pd.read_csv(project / "data/train.csv")
    test = pd.read_csv(project / "data/test.csv")
    require(TARGET in train, "train.csv is missing the target")
    require(train.drop(columns=TARGET).columns.tolist() == test.columns.tolist(), "Train/test schema differs")
    require(train.id.is_unique and test.id.is_unique, "Duplicate id")
    require(not train.id.isin(test.id).any(), "Train/test ids overlap")
    require(not train.isna().any().any() and not test.isna().any().any(), "Missing values found")
    require(set(train[TARGET].unique()) == {"No", "Yes"}, "Expected No/Yes labels")
    y = train[TARGET].map({"No": 0, "Yes": 1}).astype("int8")
    require(y.value_counts().min() >= 10, "Too few rows per class for nested five-fold validation")
    folds = list(StratifiedKFold(5, shuffle=True, random_state=42).split(train, y))
    sample = project / "data/sample_submission.csv"
    if sample.exists():
        sample_frame = pd.read_csv(sample)
        require(sample_frame.columns.tolist() == ["id", TARGET], "Unexpected submission columns")
        require(sample_frame.id.is_unique and len(sample_frame) == len(test), "Invalid sample submission ids")
        require(pd.Index(test.id).isin(sample_frame.id).all(), "Sample submission has different ids")
    return train, test, y, folds

def read_run(run_dir, model_name, train, test, y, folds, expected_auc=None):
    """Verify saved OOF, test averages, and the actual five validation splits."""
    run_dir = Path(run_dir)
    config_path = run_dir / "config.json"
    if not config_path.is_file():
        raise FileNotFoundError(f"找不到 {config_path}；请把 BASELINE_DIR 指向原训练结果目录。")
    config = json.loads(config_path.read_text())
    require(config.get("folds") == 5 and config.get("seed") == 42, "Run does not use V3 folds/seed")
    require(model_name in config.get("models", []), f"This run does not contain {model_name}")
    require(config.get("train_rows") == len(train) and config.get("test_rows") == len(test), "Dataset sizes differ")
    oof_frame = align_csv(run_dir / f"oof_{model_name}.csv", train.id, ["target", "prediction"])
    test_frame = align_csv(run_dir / f"submission_{model_name}.csv", test.id, [TARGET])
    require(np.array_equal(oof_frame.target.to_numpy(), y.to_numpy()), "OOF labels do not match current train.csv")
    oof = probabilities(oof_frame.prediction, len(train), "OOF")
    test_pred = probabilities(test_frame[TARGET], len(test), "Test")
    reconstructed = np.full(len(train), np.nan)
    coverage = np.zeros(len(train), dtype=np.int8)
    test_sum = np.zeros(len(test))
    fold_records = []
    for fold, (_, vi) in enumerate(folds, start=1):
        path = run_dir / f"{model_name}_fold{fold}_predictions.npz"
        if not path.is_file():
            raise FileNotFoundError(f"缺少 {path}；无法核对原五折，不会静默混用其他划分。")
        with np.load(path, allow_pickle=False) as saved:
            require(np.array_equal(saved["valid_indices"], vi), f"Fold {fold}: split differs")
            require(np.array_equal(saved["valid_ids"], train.id.iloc[vi]), f"Fold {fold}: train order/ids differ")
            require(np.array_equal(saved["test_ids"], test.id), f"Fold {fold}: test order/ids differ")
            vp = probabilities(saved["valid_prediction"], len(vi), f"Fold {fold} OOF")
            tp = probabilities(saved["test_prediction"], len(test), f"Fold {fold} test")
            reconstructed[vi] = vp
            coverage[vi] += 1
            test_sum += tp
            fold_records.append({"fold": fold, "auc": float(roc_auc_score(y.iloc[vi], vp))})
    require((coverage == 1).all(), "OOF coverage is not exactly once per row")
    require(np.allclose(reconstructed, oof, atol=1e-12, rtol=0), "OOF CSV differs from fold predictions")
    require(np.allclose(test_sum / 5, test_pred, atol=1e-12, rtol=0), "Test CSV is not this run's five-fold average")
    score = float(roc_auc_score(y, oof))
    if expected_auc is not None:
        require(abs(score - expected_auc) < 1e-8,
                f"旧 OOF AUC 是 {score:.10f}，不是本次预期的 {expected_auc:.10f}；请检查目录版本。")
    return {"oof": oof, "test": test_pred, "auc": score, "fold_scores": pd.DataFrame(fold_records)}

def save_json(path: Path, value: dict) -> None:
    """Atomic writes prevent incomplete metadata from being marked complete."""
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")
    temporary.replace(path)

def package_versions() -> dict:
    return {name: importlib.metadata.version(name)
            for name in ("numpy", "pandas", "scikit-learn", "lightgbm")}

def check_source_files(bundle: dict) -> None:
    require(input_hashes(bundle["project"]) == bundle["hashes"],
            "原始 CSV 已改变，请停止并核对，不要混用不同版本的数据。")
    require(file_hash(bundle["compare_dir"] / "continuation_summary.json") == bundle["summary_sha256"],
            "04 的比较摘要已改变；请重新加载并核对来源。")

def load_v4(project: Path, compare_dir: Path,
            expected_oof: float | None = 0.94598624) -> dict:
    """Read ONLY the explicitly selected V4 run. Never guess the newest folder."""
    project, compare_dir = Path(project), Path(compare_dir)
    summary_file = compare_dir / "continuation_summary.json"
    if not summary_file.is_file():
        raise FileNotFoundError(f"找不到 {summary_file}。请核对 04 第 5 节打印的比较目录。")
    summary = json.loads(summary_file.read_text(encoding="utf-8"))
    require(summary.get("selected_by_development_oof", {}).get("candidate") == "blend_cat50_lgb50",
            "这个目录不是截图中的 04 的 50/50 最优融合，不能悄悄换成别的运行。")
    train, test, y, folds = load_project(project)
    hashes = input_hashes(project)
    require(summary.get("input_sha256") == hashes,
            "当前数据哈希与 04 记录不一致。请恢复当时的数据，不要跳过检查。")
    cat_dir = Path(summary["baseline_dir"])
    lgb_dir = Path(summary["lightgbm_dir"])
    fingerprint = lgb_dir / "data_fingerprints.json"
    if not fingerprint.is_file():
        raise FileNotFoundError(f"找不到 {fingerprint}。请保留 04 的原 LightGBM 输出目录。")
    require(json.loads(fingerprint.read_text()) == hashes, "04 的 LightGBM 数据版本不一致。")
    cat = read_run(cat_dir, "catboost", train, test, y, folds)
    lgb4 = read_run(lgb_dir, "lightgbm", train, test, y, folds)
    base = {"oof": 0.5 * cat["oof"] + 0.5 * lgb4["oof"],
            "test": 0.5 * cat["test"] + 0.5 * lgb4["test"]}
    saved_oof = align_csv(compare_dir / "oof_blend_cat50_lgb50.csv", train.id,
                          ["target", "prediction"])
    saved_test = align_csv(compare_dir / "submission_next_candidate.csv", test.id, [TARGET])
    require(np.array_equal(saved_oof.target.to_numpy(), y.to_numpy()), "04 融合 OOF 标签不一致。")
    require(np.allclose(saved_oof.prediction, base["oof"], atol=1e-12, rtol=0),
            "04 的融合 OOF 不等于两个来源模型的 50/50。")
    require(np.allclose(saved_test[TARGET], base["test"], atol=1e-12, rtol=0),
            "04 的候选提交不等于来源模型的 50/50。")
    base["auc"] = float(roc_auc_score(y, base["oof"]))
    require(abs(base["auc"] - summary["selected_by_development_oof"]["oof_auc"]) < 1e-10,
            "04 摘要分数与保存的预测不一致。")
    if expected_oof is not None:
        require(abs(base["auc"] - expected_oof) < 1e-8,
                f"载入的融合 OOF {base['auc']:.10f} 与截图的 {expected_oof:.8f} 不一致。")
    bundle = {"project": project, "compare_dir": compare_dir, "cat_dir": cat_dir,
              "lgb04_dir": lgb_dir, "hashes": hashes, "summary_sha256": file_hash(summary_file),
              "train": train, "test": test, "y": y, "folds": folds,
              "base": base, "models": {"catboost": cat, "lgb04": lgb4}}
    check_source_files(bundle)
    print("04 结果核对通过。不会重训旧 CatBoost 或旧 LightGBM。")
    print("CatBoost 来源：", cat_dir)
    print("LightGBM 来源：", lgb_dir)
    print(f"当前 04 融合 OOF：{base['auc']:.10f}")
    print("当前已报告公榜：0.94618（来自你的截图，不是本程序新跑出的成绩）")
    return bundle

def submission_frame(bundle: dict, pred: np.ndarray) -> pd.DataFrame:
    test = bundle["test"]
    pred = probabilities(pred, len(test), "Submission")
    frame = pd.DataFrame({"id": test.id.to_numpy(), TARGET: pred})
    sample = bundle["project"] / "data/sample_submission.csv"
    if sample.is_file():
        ids = pd.read_csv(sample, usecols=["id"])
        frame = ids.merge(frame, on="id", how="left", validate="one_to_one", sort=False)
    probabilities(frame[TARGET], len(test), "Ordered submission")
    return frame

V4_FEATURE_SOURCE = 'from __future__ import annotations\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport sys\nimport types\nfrom datetime import datetime\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import roc_auc_score\nfrom sklearn.model_selection import StratifiedKFold\n\nTARGET = "Will_Buy_EV"\nNUMERIC = ["Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",\n           "Charging_Stations_Near_Home", "Charging_Stations_Near_Work",\n           "Environmental_Concern_Level"]\n\n\ndef features(frame, model_name):\n    """Row-local features only: no labels or full-data target statistics."""\n    x = frame.drop(columns=["id", TARGET], errors="ignore").copy()\n    for col, scale, positions in [("Age", 1, 2), ("Annual_Income_USD", 1, 6),\n                                 ("Daily_Commute_km", 10, 3),\n                                 ("Charging_Stations_Near_Home", 1, 2),\n                                 ("Charging_Stations_Near_Work", 1, 2)]:\n        values = np.rint(frame[col].to_numpy() * scale).astype(np.int64)\n        for digit in range(positions):\n            x[f"{col}_digit_{digit}"] = (values // 10**digit % 10).astype(np.int8)\n    income = np.rint(frame["Annual_Income_USD"]).astype(np.int64)\n    x["income_last_2"] = income % 100\n    x["income_last_3"] = income % 1000\n    if model_name == "catboost":\n        for col in NUMERIC:\n            scale = 10 if col == "Daily_Commute_km" else 1\n            x[f"{col}_as_category"] = np.rint(frame[col] * scale).astype(np.int64).astype(str)\n    else:\n        x["income_prefix_100"] = income // 100\n        x["income_prefix_1000"] = income // 1000\n    return x\n\ndef encode_fold(train_x, train_y, valid_x, test_x, seed, smoothing=1.0):\n    """Inner OOF for training TE; outer-training-only maps for validation/test."""\n    a, b, c = [d.copy().reset_index(drop=True) for d in (train_x, valid_x, test_x)]\n    labels = np.asarray(train_y)\n    inner = list(StratifiedKFold(5, shuffle=True, random_state=seed).split(a, labels))\n    cats = a.select_dtypes(include=["object", "string", "category"]).columns.tolist()\n    keys = NUMERIC + cats + ["income_prefix_100", "income_prefix_1000"]\n    for col in keys:\n        encoded = np.full(len(a), np.nan)\n        for fit_idx, hold_idx in inner:\n            subset = pd.DataFrame({"key": a[col].iloc[fit_idx].to_numpy(), "y": labels[fit_idx]})\n            stats = subset.groupby("key")["y"].agg(["sum", "count"])\n            prior = labels[fit_idx].mean()\n            mapping = (stats["sum"] + smoothing * prior) / (stats["count"] + smoothing)\n            encoded[hold_idx] = a[col].iloc[hold_idx].map(mapping).fillna(prior).to_numpy()\n        stats = pd.DataFrame({"key": a[col], "y": labels}).groupby("key")["y"].agg(["sum", "count"])\n        prior = labels.mean()\n        mapping = (stats["sum"] + smoothing * prior) / (stats["count"] + smoothing)\n        a[f"{col}_te"] = encoded\n        b[f"{col}_te"] = b[col].map(mapping).fillna(prior).to_numpy()\n        c[f"{col}_te"] = c[col].map(mapping).fillna(prior).to_numpy()\n        assert np.isfinite(encoded).all()\n    # Label-free frequency: exclude a training row\'s own count contribution.\n    counts = a["Annual_Income_USD"].value_counts()\n    a["income_frequency"] = (a["Annual_Income_USD"].map(counts) - 1) / (len(a) - 1)\n    for frame in (b, c):\n        frame["income_frequency"] = frame["Annual_Income_USD"].map(counts).fillna(0) / len(a)\n    for col in cats:\n        dtype = pd.CategoricalDtype(categories=pd.unique(a[col]))\n        for frame in (a, b, c):\n            frame[col] = frame[col].astype(dtype)\n    return a, b, c\n'


# 单一实验：相对于 04，仅将内层/外层目标编码的平滑强度从 1 改到 10。
V4_SPEC = {
    "n_estimators": 5000, "learning_rate": 0.05,
    "max_depth": 5, "num_leaves": 31,
    "min_child_samples": 100, "reg_lambda": 5.0,
    "early_stopping_rounds": 200, "te_smoothing": 1.0,
}
V6_SPEC = {**V4_SPEC, "te_smoothing": 10.0}
REFERENCE = "v4_cat50_lgb04_50"
CANDIDATE = "v6_cat50_lgb06_s10_50"


def runtime_code_fingerprint(*functions) -> str:
    """Hash executable logic, ignoring notebook execution filenames/line numbers."""
    def normalize(code):
        return {
            "bytecode": code.co_code.hex(),
            "constants": [normalize(c) if isinstance(c, types.CodeType) else repr(c)
                          for c in code.co_consts],
            "names": code.co_names, "variables": code.co_varnames,
            "freevars": code.co_freevars, "cellvars": code.co_cellvars,
            "argc": code.co_argcount, "posonly": code.co_posonlyargcount,
            "kwonly": code.co_kwonlyargcount, "flags": code.co_flags,
            "exception_table": getattr(code, "co_exceptiontable", b"").hex(),
        }
    values = [{"name": f.__name__, "code": normalize(f.__code__),
               "defaults": repr(f.__defaults__), "kwdefaults": repr(f.__kwdefaults__)}
              for f in functions]
    return hashlib.sha256(json.dumps(values, sort_keys=True).encode()).hexdigest()


def check_environment(bundle: dict) -> dict:
    try:
        import lightgbm as lgb
    except (ImportError, OSError) as error:
        raise RuntimeError(
            "当前内核不能导入 LightGBM。请切回成功运行 04/05 的内核。"
            "确实缺包时在此 notebook 单独运行 %pip install lightgbm，再重启内核。"
        ) from error
    path = bundle["lgb04_dir"] / "package_versions.json"
    if not path.is_file():
        raise FileNotFoundError(f"缺少 {path}；请保留完整的 04 输出。")
    old_versions = json.loads(path.read_text())
    current = package_versions()
    require(current == old_versions,
            f"当前环境版本与 04 不同，无法把变化仅归因于平滑参数。请切回原内核。\n"
            f"04：{old_versions}\n当前：{current}")
    return current


def train_lgb06(bundle: dict, threads: int = -1, spec: dict | None = None) -> tuple:
    """A new five-fold LightGBM configuration. Resume COMPLETE folds only."""
    try:
        import lightgbm as lgb
    except (ImportError, OSError) as error:
        raise RuntimeError("当前内核不能导入 lightgbm。请使用已成功运行 04 的内核；缺包时用 %pip install lightgbm。") from error
    spec = dict(V6_SPEC if spec is None else spec)
    require(spec == V6_SPEC, "本轮只执行预设的 smoothing=10、depth=5 配置，不做网格搜索。")
    require(spec["num_leaves"] <= 2 ** spec["max_depth"], "叶子数与深度配置不一致。")
    check_source_files(bundle)
    current_versions = package_versions()
    old_versions_file = bundle["lgb04_dir"] / "package_versions.json"
    if not old_versions_file.is_file():
        raise FileNotFoundError(f"缺少 {old_versions_file}；请保留完整的 04 输出。")
    old_versions = json.loads(old_versions_file.read_text())
    require(current_versions == old_versions,
            f"当前环境版本与 04 不同，不能把差异全归因于目标编码平滑强度。请切回运行 04 的环境。\n04: {old_versions}\n当前: {current_versions}")
    experiment = {"models": ["lightgbm"], "folds": 5, "seed": 42,
                  "train_rows": len(bundle["train"]), "test_rows": len(bundle["test"]),
                  "spec": spec, "threads": threads, "input_sha256": bundle["hashes"],
                  "package_versions": current_versions, "python_version": sys.version.split()[0],
                  "runtime_code_sha256": runtime_code_fingerprint(features, encode_fold, train_lgb06),
                  "feature_source_sha256": hashlib.sha256(V4_FEATURE_SOURCE.encode()).hexdigest(),
                  "trainer_revision": "v6-smoothing10-depth5-fixedblend-1", "v4_summary_sha256": bundle["summary_sha256"]}
    signature = hashlib.sha256(json.dumps(experiment, sort_keys=True).encode()).hexdigest()[:16]
    out = bundle["project"] / "outputs" / f"v6_lightgbm_s{spec['te_smoothing']:g}_{signature}"
    out.mkdir(parents=True, exist_ok=True)
    config_file = out / "config.json"
    if config_file.is_file():
        require(json.loads(config_file.read_text()) == experiment, "已有缓存配置不同，停止以防混用。")
    else:
        require(not any(out.iterdir()), "新目录中有未标记文件，不能安全复用。")
        save_json(config_file, experiment)
        save_json(out / "data_fingerprints.json", bundle["hashes"])
        save_json(out / "package_versions.json", current_versions)
        (out / "feature_code.py").write_text(V4_FEATURE_SOURCE, encoding="utf-8")
    require(file_hash(out / "feature_code.py") == experiment["feature_source_sha256"],
            "缓存中的特征代码已改变，不能复用。")
    require(json.loads((out / "data_fingerprints.json").read_text()) == bundle["hashes"],
            "缓存的数据指纹已改变，不能复用。")
    require(json.loads((out / "package_versions.json").read_text()) == current_versions,
            "缓存的环境版本记录不一致。")
    print("06 LightGBM 目录：", out)
    print("仅重训新配置；已完整保存且通过核对的折会跳过。中断那一折从头开始。")
    train, test, y, folds = bundle["train"], bundle["test"], bundle["y"], bundle["folds"]
    x, xt = features(train, "lightgbm"), features(test, "lightgbm")
    require(x.columns.tolist() == xt.columns.tolist(), "训练和测试特征不一致。")
    oof, test_sum = np.full(len(train), np.nan), np.zeros(len(test))
    records = []
    coverage = np.zeros(len(train), dtype=np.int8)
    for fold, (ti, vi) in enumerate(folds, 1):
        pred_file = out / f"lightgbm_fold{fold}_predictions.npz"
        model_file = out / f"lightgbm_fold{fold}.txt"
        index_file = out / f"lightgbm_fold{fold}_train_indices.npy"
        marker_file = out / f"fold{fold}_complete.json"
        if marker_file.is_file():
            marker = json.loads(marker_file.read_text())
            require(marker.get("experiment_signature") == signature and marker.get("fold") == fold,
                    "已完成折标记不是本次实验。")
            require(set(marker.get("file_sha256", {})) == {model_file.name, index_file.name, pred_file.name},
                    "已完成折标记缺少必要文件，不能安全复用。")
            for filename, checksum in marker["file_sha256"].items():
                path = out / filename
                require(path.is_file() and file_hash(path) == checksum,
                        f"第 {fold} 折缓存缺失/被改动：{filename}。停止，不混用。")
            require(np.array_equal(np.load(index_file, allow_pickle=False), ti), "缓存训练折不一致。")
            with np.load(pred_file, allow_pickle=False) as saved:
                require(np.array_equal(saved["valid_indices"], vi), "缓存验证折不一致。")
                require(np.array_equal(saved["valid_ids"], train.id.iloc[vi]), "缓存验证 ID 不一致。")
                require(np.array_equal(saved["test_ids"], test.id), "缓存测试 ID 不一致。")
                vp = probabilities(saved["valid_prediction"], len(vi), "Cached OOF")
                tp = probabilities(saved["test_prediction"], len(test), "Cached test")
            best = int(marker["best_trees"])
            print(f"复用已完成 Fold {fold}/5，best_trees={best}")
        else:
            print(f"\n========== 新 LightGBM | Fold {fold}/5 ==========", flush=True)
            a, b, c = encode_fold(x.iloc[ti], y.iloc[ti], x.iloc[vi], xt,
                                  seed=1041 + fold, smoothing=spec["te_smoothing"])
            bins = [1023 if col == "Annual_Income_USD" else 255 for col in a.columns]
            model_params = {key: value for key, value in spec.items()
                            if key not in {"early_stopping_rounds", "te_smoothing"}}
            model = lgb.LGBMClassifier(
                **model_params, max_bin_by_feature=bins, objective="binary", metric="auc",
                random_state=41 + fold, n_jobs=threads, verbosity=-1,
                deterministic=True, force_col_wise=True,
            )
            model.fit(a, y.iloc[ti], eval_set=[(b, y.iloc[vi])], eval_metric="auc",
                      callbacks=[lgb.early_stopping(spec["early_stopping_rounds"], first_metric_only=True),
                                 lgb.log_evaluation(400)])
            require(np.array_equal(model.classes_, [0, 1]), "正类概率列不符合预期。")
            vp = probabilities(model.predict_proba(b)[:, 1], len(vi), "New OOF")
            tp = probabilities(model.predict_proba(c)[:, 1], len(test), "New test")
            best = int(model.best_iteration_)
            model.booster_.save_model(str(model_file))
            np.save(index_file, ti)
            with pred_file.with_suffix(".npz.tmp").open("wb") as handle:
                np.savez_compressed(handle, valid_indices=vi, valid_ids=train.id.iloc[vi].to_numpy(),
                                    valid_prediction=vp, test_ids=test.id.to_numpy(), test_prediction=tp)
            pred_file.with_suffix(".npz.tmp").replace(pred_file)
            save_json(marker_file, {"fold": fold, "best_trees": best, "experiment_signature": signature,
                      "file_sha256": {p.name: file_hash(p) for p in (model_file, index_file, pred_file)}})
            del a, b, c, model
            gc.collect()
        oof[vi] = vp
        coverage[vi] += 1
        test_sum += tp
        score = float(roc_auc_score(y.iloc[vi], vp))
        records.append({"model": "lightgbm", "fold": fold, "auc": score, "best_trees": best})
        pd.DataFrame(records).to_csv(out / "fold_scores.csv", index=False)
        print(f"Fold AUC: {score:.10f}", flush=True)
    require((coverage == 1).all(), "新 OOF 每行必须恰好被验证一次。")
    probabilities(oof, len(train), "Completed OOF")
    pd.DataFrame({"id": train.id.to_numpy(), "target": y.to_numpy(), "prediction": oof}).to_csv(
        out / "oof_lightgbm.csv", index=False)
    pd.DataFrame({"id": test.id.to_numpy(), TARGET: test_sum / len(folds)}).to_csv(
        out / "submission_lightgbm.csv", index=False)
    check_source_files(bundle)
    result = read_run(out, "lightgbm", train, test, y, folds)
    save_json(out / "summary.json", {"oof_auc": result["auc"], "spec": spec, "inner_seed_rule": "1041 + fold",
              "note": "Single-model development OOF only; final blend comparison is a separate step."})
    print(f"新 LightGBM 单模型 OOF：{result['auc']:.10f}；请继续与 04 融合比较。")
    return out, result


def compare_v06(bundle: dict, new: dict, new_dir: Path) -> tuple:
    """Only two submission candidates: old 50/50 vs new 50/50. No weight search."""
    check_source_files(bundle)
    # Re-read all new fold predictions and match IDs/splits before mixing results.
    checked = read_run(new_dir, "lightgbm", bundle["train"], bundle["test"],
                       bundle["y"], bundle["folds"])
    for part in ("oof", "test"):
        require(np.allclose(checked[part], new[part], atol=1e-12, rtol=0),
                "内存中新 LightGBM 预测与保存文件不一致。")
    new = checked
    y, folds, base = bundle["y"], bundle["folds"], bundle["base"]
    cat, lgb04 = bundle["models"]["catboost"], bundle["models"]["lgb04"]
    blend = {part: 0.5 * cat[part] + 0.5 * new[part] for part in ("oof", "test")}
    base_fold = [float(roc_auc_score(y.iloc[vi], base["oof"][vi])) for _, vi in folds]
    records, per_fold = [], []
    for name, predictions in ((REFERENCE, base), (CANDIDATE, blend)):
        probabilities(predictions["oof"], len(y), name + " OOF")
        probabilities(predictions["test"], len(bundle["test"]), name + " test")
        score = float(roc_auc_score(y, predictions["oof"]))
        deltas = []
        for fold, (_, vi) in enumerate(folds, 1):
            fold_auc = float(roc_auc_score(y.iloc[vi], predictions["oof"][vi]))
            delta = fold_auc - base_fold[fold - 1]
            deltas.append(delta)
            per_fold.append({"candidate": name, "fold": fold, "auc": fold_auc,
                             "delta_vs_v4_blend": delta})
        records.append({"candidate": name, "oof_auc": score,
                        "delta_vs_v4_blend": score - base["auc"],
                        "improved_folds_vs_v4": sum(d > 0 for d in deltas),
                        "worst_fold_delta_vs_v4": min(deltas)})
    comparison = pd.DataFrame(records).sort_values("oof_auc", ascending=False, kind="stable").reset_index(drop=True)
    by_fold = pd.DataFrame(per_fold)
    diagnostic = pd.DataFrame([
        {"model": "lgb04_smoothing1", "oof_auc": lgb04["auc"], "delta_vs_lgb04": 0.0},
        {"model": "lgb06_smoothing10", "oof_auc": new["auc"], "delta_vs_lgb04": new["auc"] - lgb04["auc"]},
    ])
    new_row = next(row for row in records if row["candidate"] == CANDIDATE)
    delta = float(new_row["delta_vs_v4_blend"])
    same_test = bool(np.array_equal(blend["test"], base["test"]))
    selected = CANDIDATE if delta > 0 else REFERENCE  # Keep 04 on an exact tie.
    out = bundle["project"] / "outputs" / ("v6_compare_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    out.mkdir(parents=True, exist_ok=False)
    comparison.to_csv(out / "comparison.csv", index=False)
    by_fold.to_csv(out / "comparison_by_fold.csv", index=False)
    diagnostic.to_csv(out / "lightgbm_diagnostic.csv", index=False)
    pd.DataFrame({"id": bundle["train"].id.to_numpy(), "target": y.to_numpy(),
                  "prediction": blend["oof"]}).to_csv(out / "oof_v06_fixed_blend.csv", index=False)
    submission_path = None
    # No disguised baseline CSV: losing experiments do NOT create a candidate submission.
    if selected == CANDIDATE and not same_test:
        submission_path = out / "submission_v06_smooth10_cat50_lgb50.csv"
        submission_frame(bundle, blend["test"]).to_csv(submission_path, index=False)
    summary = {
        "experiment": "V6: TE smoothing 1 -> 10, all model settings as V4, fixed Cat/LGB 50/50",
        "selected_by_development_oof": selected,
        "reference_oof_auc": base["auc"], "new_blend_oof_auc": new_row["oof_auc"],
        "delta_vs_v4_blend": delta, "improved_folds_vs_v4": new_row["improved_folds_vs_v4"],
        "worst_fold_delta_vs_v4": new_row["worst_fold_delta_vs_v4"],
        "identical_to_v4_predictions": same_test,
        "candidate_submission": str(submission_path) if submission_path is not None else None,
        "status": "REVIEW_NEW_CANDIDATE" if submission_path is not None else "KEEP_V4",
        "spec_v4": V4_SPEC, "spec_v6": V6_SPEC,
        "v4_compare_dir": str(bundle["compare_dir"]), "lgb06_dir": str(new_dir),
        "input_sha256": bundle["hashes"], "v4_summary_sha256": bundle["summary_sha256"],
        "candidates": records,
        "prior_scores_from_user_screenshots": {
            "v4_50_50_public": 0.94618, "v5_old_models_25_75_public": 0.94615,
            "v5_old_models_25_75_oof_rounded": 0.946021,
        },
        "validation_note": "Development OOF reused for early stopping and sequential experiment selection; not an independent test. Fold direction counts are not significance tests.",
        "public_note": "Prior public feedback informed this experiment. No new Kaggle score is known; no automatic submission or replacement of prior submissions.",
    }
    save_json(out / "summary.json", summary)
    print("【主比较：看这两行，不要把单模型诊断当成提交选择】")
    print(comparison.to_string(index=False, float_format=lambda value: f"{value:.8f}"))
    print("\n结果目录：", out)
    if submission_path is not None:
        print("结论：新固定融合的开发 OOF 略高或更高，已导出待评估候选；并未验证公榜提分。")
        print("候选 CSV：", submission_path)
        print("先查看总表和逐折变化。保留 04 的 0.94618 提交，不自动替换。")
    else:
        print("结论：保留 04。本轮没有导出新的提交候选，不需要找 CSV 或重复提交。")
    print("OOF 分数不是 Kaggle 分数；即使五折都改善，也不保证公榜/私榜更高。")
    return out, comparison, by_fold, diagnostic, summary


## 2. 检查路径、旧预测和环境（不训练）

默认目录来自你提供的 04 截图。若路径不存在，恢复旧文件或修正 `PROJECT` / `V4_COMPARE_DIR`；不要删除核对代码。正常情况下旧融合 OOF 约为 `0.94598624`。

In [2]:
PROJECT = Path.cwd().parent  # 项目根目录（notebooks/ 的上一级）
V4_COMPARE_DIR = PROJECT / "outputs/v4_compare_20260930_163707_027699"

# 自动从指定 04 摘要读取 CatBoost、旧 LightGBM 的输出目录，并核对每折预测与数据。
bundle = load_v4(PROJECT, V4_COMPARE_DIR)
versions = check_environment(bundle)
old_config = json.loads((bundle["lgb04_dir"] / "config.json").read_text())
THREADS = int(old_config.get("threads", -1))
print("当前 Python：", sys.executable)
print("依赖版本核对通过：", versions)
print("沿用 04 线程设置：", THREADS)
print("本轮只改 smoothing：1.0 -> 10.0；max_depth=5；融合固定 50/50。")
print("检查完成。继续运行第 3 节会开始训练。")


04 结果核对通过。不会重训旧 CatBoost 或旧 LightGBM。
CatBoost 来源： /Users/yw/unsw/Kaggle/outputs/v3_20260925_180457_278867
LightGBM 来源： /Users/yw/unsw/Kaggle/outputs/v4_lightgbm_20260930_163526_079778
当前 04 融合 OOF：0.9459862434
当前已报告公榜：0.94618（来自你的截图，不是本程序新跑出的成绩）
当前 Python： /opt/miniconda3/envs/dl/bin/python
依赖版本核对通过： {'numpy': '2.4.6', 'pandas': '3.0.6', 'scikit-learn': '1.9.1', 'lightgbm': '4.7.0'}
沿用 04 线程设置： -1
本轮只改 smoothing：1.0 -> 10.0；max_depth=5；融合固定 50/50。
检查完成。继续运行第 3 节会开始训练。


## 3. 训练 06 的 LightGBM 五折

**运行下一格会启动训练。无需把 False 改成 True。** 原来的 CatBoost、04 的 LightGBM 都不会重训。保留早停；5000 是轮数上限，不是保证每折都跑满。

完整折可复用，中断的折从头训练。重启内核后重新运行本文件即可核对并复用；不要删除 `v6_lightgbm_s10_*` 目录中的文件，也不要同时在两个内核运行同一实验。

In [3]:
# 运行本格就会训练；不要再次运行旧的 04/05 训练单元格。
# 同配置已完成的 06 折会校验后复用；中断那一折从头重新训练。
lgb06_dir, lgb06_result = train_lgb06(bundle, threads=THREADS)
print("06 的五折已完成。继续运行第 4 节，查看固定融合比较。")


06 LightGBM 目录： /Users/yw/unsw/Kaggle/outputs/v6_lightgbm_s10_762ed6fc03204b55
仅重训新配置；已完整保存且通过核对的折会跳过。中断那一折从头开始。

========== 新 LightGBM | Fold 1/5 ==========


/opt/miniconda3/envs/dl/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[400]	valid_0's auc: 0.945067
Early stopping, best iteration is:
[556]	valid_0's auc: 0.945083
Evaluated only: auc
Fold AUC: 0.9450826627

========== 新 LightGBM | Fold 2/5 ==========


/opt/miniconda3/envs/dl/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[400]	valid_0's auc: 0.945589
Early stopping, best iteration is:
[440]	valid_0's auc: 0.945611
Evaluated only: auc
Fold AUC: 0.9456113574

========== 新 LightGBM | Fold 3/5 ==========


...

Training until validation scores don't improve for 200 rounds
[400]	valid_0's auc: 0.94675
Early stopping, best iteration is:
[397]	valid_0's auc: 0.946753
Evaluated only: auc
Fold AUC: 0.9467528746

========== 新 LightGBM | Fold 4/5 ==========


/opt/miniconda3/envs/dl/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[400]	valid_0's auc: 0.946224
Early stopping, best iteration is:
[390]	valid_0's auc: 0.946228
Evaluated only: auc
Fold AUC: 0.9462277232

========== 新 LightGBM | Fold 5/5 ==========


/opt/miniconda3/envs/dl/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 200 rounds
[400]	valid_0's auc: 0.945995
[800]	valid_0's auc: 0.945998
Early stopping, best iteration is:
[628]	valid_0's auc: 0.946027
Evaluated only: auc
Fold AUC: 0.9460274672
新 LightGBM 单模型 OOF：0.9459343244；请继续与 04 融合比较。
06 的五折已完成。继续运行第 4 节，查看固定融合比较。


## 4. 比较固定 50/50 融合，并给出下一步

只比较旧 04 的 50/50 与新 06 的 50/50。单模型分数另外显示，不混入候选池。本地对照是 **0.94598624 左右**，不是公榜 **0.94618**。

新融合 OOF 未超过旧融合时，**不导出新的提交候选**。超过时仅导出一个名称明确的 `submission_v06_smooth10_cat50_lgb50.csv`；它仍然只是待评估候选，不代表必须提交，不自动替换旧公榜最佳。先看下面主表和逐折变化。

In [4]:
comparison_dir, comparison, by_fold, diagnostic, result_summary = compare_v06(
    bundle, lgb06_result, lgb06_dir
)

# 主表：先看这两行。数字使用 8 位小数，避免小变化在显示中被舍去。
display(comparison.style.format({
    "oof_auc": "{:.8f}", "delta_vs_v4_blend": "{:+.8f}",
    "worst_fold_delta_vs_v4": "{:+.8f}",
}))
print("逐折变化（全部相对旧 04 的 50/50 融合）：")
display(by_fold.pivot(index="fold", columns="candidate", values="delta_vs_v4_blend")
        .style.format("{:+.8f}"))
print("单模型诊断（仅解释结果，不参与本轮提交选择）：")
display(diagnostic.style.format({"oof_auc": "{:.8f}", "delta_vs_lgb04": "{:+.8f}"}))


【主比较：看这两行，不要把单模型诊断当成提交选择】
            candidate    oof_auc  delta_vs_v4_blend  improved_folds_vs_v4  worst_fold_delta_vs_v4
v6_cat50_lgb06_s10_50 0.94598801         0.00000177                     2             -0.00003609
    v4_cat50_lgb04_50 0.94598624         0.00000000                     0              0.00000000

结果目录： /Users/yw/unsw/Kaggle/outputs/v6_compare_20260930_184804_018261
结论：新固定融合的开发 OOF 略高或更高，已导出待评估候选；并未验证公榜提分。
候选 CSV： /Users/yw/unsw/Kaggle/outputs/v6_compare_20260930_184804_018261/submission_v06_smooth10_cat50_lgb50.csv
先查看总表和逐折变化。保留 04 的 0.94618 提交，不自动替换。
OOF 分数不是 Kaggle 分数；即使五折都改善，也不保证公榜/私榜更高。


,candidate,oof_auc,delta_vs_v4_blend,improved_folds_vs_v4,worst_fold_delta_vs_v4
0,v6_cat50_lgb06_s10_50,0.94598801,+0.00000177,2,-0.00003609
1,v4_cat50_lgb04_50,0.94598624,+0.00000000,0,+0.00000000


逐折变化（全部相对旧 04 的 50/50 融合）：


candidate,v4_cat50_lgb04_50,v6_cat50_lgb06_s10_50
fold,,
1,+0.00000000,+0.00006192
2,+0.00000000,+0.00001036
3,+0.00000000,-0.00003139
4,+0.00000000,-0.00003609
5,+0.00000000,-0.00000086


单模型诊断（仅解释结果，不参与本轮提交选择）：


,model,oof_auc,delta_vs_lgb04
0,lgb04_smoothing1,0.94594638,+0.00000000
1,lgb06_smoothing10,0.94593432,-0.00001206


## 保存了什么？

`outputs/v6_lightgbm_s10_<配置指纹>/` 是训练记录，包含五折模型、每折预测、索引、完成标记、特征代码与版本配置。模型文件本身不包含整个目标编码流程；以后重新推理必须配合同目录 `feature_code.py`、配置、折索引和相同训练数据重建映射。

`outputs/v6_compare_<时间戳>/` 是本轮比较结果，包含主表、逐折结果、单模型诊断、06 融合 OOF、摘要，以及在开发 OOF 改善时才生成的唯一候选提交 CSV。**不要拿训练目录里的单模型 CSV 代替这里的融合候选。**

摘要 `status=KEEP_V4` 表示保留 04，无需提交；`REVIEW_NEW_CANDIDATE` 表示有新候选，先核对比较结果。整体 OOF 的微小提升、甚至每折都同向，都不构成统计显著性保证，实际公榜/私榜仍可能降低。这轮没有使用私榜标签，也没有增加独立留出测试；相同 OOF 仍被反复用于开发。